# S6. Текстові моделі на слабких мітках: TF-IDF, ембединги bge-m3, XLM-R

**Що робить ноутбук.** Навчає текстові моделі на пулі зі слабкими (LLM) мітками і зберігає їхні ймовірності для золотих повідомлень за протоколами P1, P2, P3
(спліти S5 вбудовано в ноутбук). Золоті мітки тут не використовуються взагалі, метрики рахуються потім у хмарній сесії.

Моделі: TF-IDF (слова і символьні n-грами) + LogReg; ембединги bge-m3 з кешу S4 + LogReg; дотренування `xlm-roberta-base` (3 seed; P1 і P2).
Контролі з перемішаними мітками (P1) для кожної моделі. C у LogReg підбирається на валідації слабких міток (10% пулу), епоха XLM-R обирається так само.
Перед початком ноутбук звіряє хеші текстів з індексом ембедингів S4 і проганяє самоперевірку циклу XLM-R на крихітній випадковій моделі.

**Що зробити вам.**
1. Runtime → Change runtime type → **T4 GPU**.
2. Runtime → Run all, дозволити доступ до Drive.
3. Коли завершиться, надішліть мені `MyDrive/narratives-thesis/results/06_text_models.zip`.

Орієнтовний час 1,5-2,5 години (TF-IDF і ембединги близько 20-30 хвилин на CPU, XLM-R близько 1-1,5 години на T4). Потрібен Drive з S1 і S4
(`data/raw/telegram_messages.parquet`, `data/interim/embeddings/`). Якщо сеанс обірветься, просто запустіть знову: завершені одиниці пропускаються.
Якщо часу мало, у клітинці запуску змініть `XLMR_PROTOCOLS = "P1"` (без P2) або `SEEDS = 2`; це треба буде записати у звіт.


In [ ]:
# 1. Drive, залежності, GPU, шляхи
import base64, hashlib, io, json, subprocess, sys, time, zipfile
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/narratives-thesis")
    REPO = Path("/content/repo")
except ImportError:  # запуск поза Colab (тестовий)
    BASE = Path("./narratives-thesis").resolve()
    REPO = Path("./repo").resolve()

RAW = BASE / "data" / "raw"
EMB = BASE / "data" / "interim" / "embeddings"
OUT = BASE / "results" / "06_text_models"
OUT.mkdir(parents=True, exist_ok=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "pyyaml"], check=True)
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "GPU не підключено. Runtime -> Change runtime type -> T4 GPU, потім запустіть знову."
print("Дані:", RAW, "| ембединги:", EMB, "| результати:", OUT)


In [ ]:
# 2. Код репозиторію і спліти S5 (вбудовані в ноутбук)
FILES = {'src/narr/__init__.py': '"""Код магістерської роботи: гібридні моделі (граф + текст) для виявлення дезінформаційних наративів."""\n\n__version__ = "0.1.0"\n', 'src/narr/baselines.py': '"""Текстові бейзлайни: TF-IDF (слова, символьні n-грами) + логістична регресія, дамі-моделі, повторна CV."""\nfrom __future__ import annotations\n\nimport warnings\n\nimport numpy as np\nfrom joblib import Parallel, delayed\nfrom sklearn.dummy import DummyClassifier\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression, LogisticRegressionCV\nfrom sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold\nfrom sklearn.pipeline import FeatureUnion, Pipeline\n\nMODEL_NAMES = ["majority", "stratified", "word", "char", "word_char"]\nCS = [0.1, 1, 10, 100]\n\n\ndef _word() -> TfidfVectorizer:\n    return TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=50000)\n\n\ndef _char() -> TfidfVectorizer:\n    return TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True, max_features=100000)\n\n\ndef make_model(name: str, seed: int, task: str = "binary"):\n    """Підбір C відбувається лише всередині навчальної частини (вкладена CV), тест не бачить підбору."""\n    scoring = "average_precision" if task == "binary" else "f1_macro"\n\n    def lr():\n        return LogisticRegressionCV(\n            Cs=CS, cv=StratifiedKFold(3, shuffle=True, random_state=seed), scoring=scoring,\n            class_weight="balanced", max_iter=3000, random_state=seed)\n\n    if name == "majority":\n        return DummyClassifier(strategy="most_frequent")\n    if name == "stratified":\n        return DummyClassifier(strategy="stratified", random_state=seed)\n    if name == "word":\n        return Pipeline([("tfidf", _word()), ("clf", lr())])\n    if name == "char":\n        return Pipeline([("tfidf", _char()), ("clf", lr())])\n    if name == "word_char":\n        return Pipeline([("tfidf", FeatureUnion([("w", _word()), ("c", _char())])), ("clf", lr())])\n    raise ValueError(name)\n\n\ndef _fit_predict(name: str, task: str, X: np.ndarray, y: np.ndarray, tr: np.ndarray, te: np.ndarray, seed: int):\n    with warnings.catch_warnings():\n        warnings.simplefilter("ignore")\n        m = make_model(name, seed, task)\n        m.fit(X[tr], y[tr])\n        return te, m.predict_proba(X[te])\n\n\ndef oof_proba(name: str, X: np.ndarray, y: np.ndarray, *, task: str = "binary", n_splits: int = 5,\n              n_repeats: int = 5, seed: int = 42, groups: np.ndarray | None = None, n_jobs: int = 2) -> np.ndarray:\n    """Повертає (n_repeats, N, K) ймовірності out-of-fold: кожне повідомлення передбачене моделлю, що його не бачила."""\n    jobs = []\n    for r in range(n_repeats):\n        if groups is None:\n            cv = StratifiedKFold(n_splits, shuffle=True, random_state=seed + r)\n            it = cv.split(X, y)\n        else:\n            cv = StratifiedGroupKFold(n_splits, shuffle=True, random_state=seed + r)\n            it = cv.split(X, y, groups)\n        jobs += [(r, tr, te) for tr, te in it]\n    res = Parallel(n_jobs=n_jobs)(delayed(_fit_predict)(name, task, X, y, tr, te, seed + r) for r, tr, te in jobs)\n    k = len(np.unique(y))\n    P = np.zeros((n_repeats, len(y), k))\n    for (r, _, _), (te, pr) in zip(jobs, res):\n        P[r, te, :] = pr\n    return P\n\n\ndef top_features(texts: np.ndarray, y: np.ndarray, n: int = 20, C: float = 1.0) -> tuple[list[str], list[str]]:\n    """Описова перевірка змісту: найсильніші слова за коефіцієнтами на всіх даних (не оцінка якості)."""\n    vec = _word()\n    X = vec.fit_transform(texts)\n    clf = LogisticRegression(C=C, class_weight="balanced", max_iter=3000).fit(X, y)\n    names = np.array(vec.get_feature_names_out())\n    order = np.argsort(clf.coef_[0])\n    return list(names[order[::-1][:n]]), list(names[order[:n]])\n', 'src/narr/evaluate.py': '"""Метрики, бутстреп-довірчі інтервали, парні порівняння."""\nfrom __future__ import annotations\n\nfrom typing import Callable\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import average_precision_score\n\n\ndef confusion(y: np.ndarray, pred: np.ndarray, k: int) -> np.ndarray:\n    return np.bincount(np.asarray(y) * k + np.asarray(pred), minlength=k * k).reshape(k, k)\n\n\ndef mcc_from_cm(cm: np.ndarray) -> float:\n    """Багатокласовий MCC (Городкін); для k=2 збігається з класичним."""\n    t, p = cm.sum(1).astype(float), cm.sum(0).astype(float)\n    c, s = float(np.trace(cm)), float(cm.sum())\n    den = np.sqrt(s ** 2 - (p * p).sum()) * np.sqrt(s ** 2 - (t * t).sum())\n    return float((c * s - (t * p).sum()) / den) if den > 0 else 0.0\n\n\ndef f1_per_class(cm: np.ndarray) -> np.ndarray:\n    tp = np.diag(cm).astype(float)\n    fp, fn = cm.sum(0) - tp, cm.sum(1) - tp\n    den = 2 * tp + fp + fn\n    return np.divide(2 * tp, den, out=np.zeros_like(tp), where=den > 0)\n\n\ndef binary_metrics(y: np.ndarray, p1: np.ndarray, thr: float = 0.5) -> dict:\n    pred = (p1 >= thr).astype(int)\n    cm = confusion(y, pred, 2)\n    f1 = f1_per_class(cm)\n    tp, fp, fn = cm[1, 1], cm[0, 1], cm[1, 0]\n    return {\n        "macro_f1": float(f1.mean()), "mcc": mcc_from_cm(cm),\n        "pr_auc": float(average_precision_score(y, p1)) if y.sum() > 0 else float("nan"),\n        "f1_pos": float(f1[1]), "f1_neg": float(f1[0]),\n        "precision": float(tp / (tp + fp)) if (tp + fp) else 0.0,\n        "recall": float(tp / (tp + fn)) if (tp + fn) else 0.0,\n    }\n\n\ndef multiclass_metrics(y: np.ndarray, pred: np.ndarray, k: int) -> dict:\n    cm = confusion(y, pred, k)\n    f1 = f1_per_class(cm)\n    out = {"macro_f1": float(f1.mean()), "mcc": mcc_from_cm(cm), "accuracy": float(np.trace(cm) / cm.sum())}\n    out.update({f"f1_class{i}": float(v) for i, v in enumerate(f1)})\n    return out\n\n\ndef _mean_over_repeats(fn: Callable, y: np.ndarray, P: np.ndarray, idx: np.ndarray | None = None) -> dict:\n    ms = [fn(y if idx is None else y[idx], p if idx is None else p[idx]) for p in P]\n    return {k: float(np.nanmean([m[k] for m in ms])) for k in ms[0]}\n\n\ndef bootstrap_ci(y: np.ndarray, P: np.ndarray, fn: Callable[[np.ndarray, np.ndarray], dict],\n                 n_boot: int = 1000, seed: int = 0, alpha: float = 0.05) -> pd.DataFrame:\n    """P: (R, N) або (R, N, ...) прогнози для R повторів CV. Точкова оцінка: середнє метрики по повторах на всіх повідомленнях.\n    CI: процентильний бутстреп по повідомленнях (метрика на кожній вибірці усереднюється по повторах).\n    sd_repeats: розкид точкової метрики між повторами CV (окремо від CI)."""\n    y = np.asarray(y)\n    n = len(y)\n    rng = np.random.default_rng(seed)\n    per_rep = pd.DataFrame([fn(y, p) for p in P])\n    point = per_rep.mean()\n    boots = pd.DataFrame([_mean_over_repeats(fn, y, P, rng.integers(0, n, n)) for _ in range(n_boot)])\n    return pd.DataFrame({\n        "point": point, "ci_lo": boots.quantile(alpha / 2), "ci_hi": boots.quantile(1 - alpha / 2),\n        "sd_repeats": per_rep.std(ddof=1) if len(per_rep) > 1 else np.nan,\n    }).rename_axis("metric").reset_index()\n\n\ndef paired_bootstrap_diff(y: np.ndarray, PA: np.ndarray, PB: np.ndarray, fn: Callable,\n                          n_boot: int = 1000, seed: int = 0, alpha: float = 0.05) -> pd.DataFrame:\n    """Різниця метрик A мінус B з парним бутстрепом (ті самі індекси для обох моделей)."""\n    y = np.asarray(y)\n    n = len(y)\n    rng = np.random.default_rng(seed)\n    pa, pb = _mean_over_repeats(fn, y, PA), _mean_over_repeats(fn, y, PB)\n    diffs = []\n    for _ in range(n_boot):\n        idx = rng.integers(0, n, n)\n        a, b = _mean_over_repeats(fn, y, PA, idx), _mean_over_repeats(fn, y, PB, idx)\n        diffs.append({k: a[k] - b[k] for k in a})\n    d = pd.DataFrame(diffs)\n    return pd.DataFrame({\n        "diff": {k: pa[k] - pb[k] for k in pa}, "ci_lo": d.quantile(alpha / 2), "ci_hi": d.quantile(1 - alpha / 2),\n    }).rename_axis("metric").reset_index()\n', 'src/narr/splits.py': '"""S5: протоколи оцінювання P1-P3, фільтр витоку, канальний пріор лише з train.\n\nНавчальний пул: повідомлення розміченої партії зі слабкими мітками, з яких прибрано\n  1) самі золоті повідомлення,\n  2) повідомлення з тим самим нормалізованим текстом, що й у якогось золотого,\n  3) семантичні копії золотих (cos >= 0,985 за S4, список `exclude_labeled_keys_cos985.csv`).\nЗолоті повідомлення в навчання не потрапляють ніколи (мітки золотих лише для оцінки).\n\nP1: пул, тест усі 412 золотих.\nP2: пул з датою до CUTOFF, тест золоті з датою від CUTOFF.\nP3: золоті канали ділимо на K груп (за каналами), для кожної групи тест її золоті, а з пулу прибираємо всі повідомлення її каналів.\n    (План казав «канали, яких не було в навчанні», але всі 56 золотих каналів мають слабко розмічені повідомлення, див. reports/05_protocol.md.)\n"""\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nKEY_MULT = 10_000_000  # ключ повідомлення = channel_id * KEY_MULT + message_id (як в eda.py і semantic.py)\nCUTOFF = "2025-09-01"\nSEM_THRESHOLD = 0.985\nREASONS = ("is_gold", "same_text_as_gold", "semantic_copy_of_gold")\n\n\ndef make_key(channel_id, message_id) -> np.ndarray:\n    return np.asarray(channel_id, dtype="int64") * KEY_MULT + np.asarray(message_id, dtype="int64")\n\n\ndef gold_frame(gold: pd.DataFrame, text_hash: pd.Series | None = None, binary: str = "Binary label",\n               date: str = "date_utc") -> pd.DataFrame:\n    """Золоті: key, channel_id, message_id, date_utc, y (+ text_hash, якщо передано відображення key -> хеш)."""\n    out = pd.DataFrame({"channel_id": gold["channel_id"].astype("int64"), "message_id": gold["message_id"].astype("int64"),\n                        "date_utc": pd.to_datetime(gold[date], utc=True), "y": gold[binary].astype(int)})\n    out["key"] = make_key(out["channel_id"], out["message_id"])\n    if text_hash is not None:\n        out["text_hash"] = out["key"].map(text_hash)\n    return out.sort_values("key").reset_index(drop=True)\n\n\ndef build_pool(index: pd.DataFrame, gold_keys: set, gold_hashes: set, exclude_keys: set) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """index: labeled_index (key, channel_id, message_id, date_utc, text_hash, in_gold, in_labeled, weak_narrative).\n    Повертає (pool, excluded). excluded має булеві колонки за причинами, pool це розмічені без жодної причини."""\n    lab = index[index["in_labeled"].astype(bool)].copy()\n    lab["date_utc"] = pd.to_datetime(lab["date_utc"], utc=True)\n    lab["is_gold"] = lab["key"].isin(gold_keys)\n    lab["same_text_as_gold"] = lab["text_hash"].isin(gold_hashes)\n    lab["semantic_copy_of_gold"] = lab["key"].isin(exclude_keys)\n    bad = lab[list(REASONS)].any(axis=1)\n    pool = lab.loc[~bad, ["key", "channel_id", "message_id", "date_utc", "text_hash", "weak_narrative"]].reset_index(drop=True)\n    pool["weak_narrative"] = pool["weak_narrative"].astype(bool)\n    excluded = lab.loc[bad, ["key", "channel_id", "message_id", *REASONS]].reset_index(drop=True)\n    return pool, excluded\n\n\ndef exclusion_summary(n_labeled: int, excluded: pd.DataFrame) -> dict:\n    """Скільки прибрано і чому (за кожною причиною окремо і за першою причиною в порядку REASONS)."""\n    out = {"n_labeled": int(n_labeled), "n_excluded": int(len(excluded)), "n_pool": int(n_labeled - len(excluded))}\n    for r in REASONS:\n        out[f"reason_{r}"] = int(excluded[r].sum()) if len(excluded) else 0\n    first = excluded[list(REASONS)].idxmax(axis=1) if len(excluded) else pd.Series([], dtype=object)\n    for r in REASONS:\n        out[f"first_reason_{r}"] = int((first == r).sum())\n    return out\n\n\ndef split_p1(pool: pd.DataFrame, gold: pd.DataFrame) -> dict:\n    return {"train": pool, "test": gold}\n\n\ndef split_p2(pool: pd.DataFrame, gold: pd.DataFrame, cutoff: str = CUTOFF) -> dict:\n    t = pd.Timestamp(cutoff, tz="UTC")\n    return {"train": pool[pool["date_utc"] < t].reset_index(drop=True), "test": gold[gold["date_utc"] >= t].reset_index(drop=True)}\n\n\ndef channel_folds(gold: pd.DataFrame, k: int = 5, seed: int = 42) -> pd.DataFrame:\n    """Жадібний розподіл золотих каналів по k групах. Канали йдуть від тих, що мають найбільше російськомовних, потім наративних, потім усіх повідомлень; кожен потрапляє в групу, де після додавання\n    найменша сума квадратів нормованих навантажень за трьома показниками: кількість повідомлень, кількість наративних, кількість російськомовних\n    (останнє лише якщо в gold є колонка lang). Рівні за розміром канали розставляє seed. Повертає channel_id, fold (1..k), n, n_pos, n_ru."""\n    g = gold.assign(is_ru=(gold["lang"] == "ru").astype(int) if "lang" in gold else 0)\n    ch = g.groupby("channel_id").agg(n=("key", "size"), n_pos=("y", "sum"), n_ru=("is_ru", "sum")).reset_index()\n    rng = np.random.default_rng(seed)\n    ch["tie"] = rng.permutation(len(ch))\n    ch = ch.sort_values(["n_ru", "n_pos", "n", "tie"], ascending=[False, False, False, True]).reset_index(drop=True)\n    cols = [c for c in ("n", "n_pos", "n_ru") if ch[c].sum() > 0]\n    target = {c: ch[c].sum() / k for c in cols}\n    load = {c: np.zeros(k) for c in cols}\n    fold = np.zeros(len(ch), dtype=int)\n    for i in range(len(ch)):\n        cost = sum(((load[c] + ch.loc[i, c]) / target[c]) ** 2 for c in cols)\n        j = int(np.argmin(cost))\n        fold[i] = j + 1\n        for c in cols:\n            load[c][j] += ch.loc[i, c]\n    ch["fold"] = fold\n    return ch.drop(columns="tie").sort_values(["fold", "channel_id"]).reset_index(drop=True)\n\n\ndef split_p3(pool: pd.DataFrame, gold: pd.DataFrame, folds: pd.DataFrame) -> dict:\n    """Словник {fold: {train, test}}. Для групи f з пулу прибрано всі повідомлення каналів цієї групи."""\n    out = {}\n    for f in sorted(folds["fold"].unique()):\n        chs = set(folds.loc[folds["fold"] == f, "channel_id"])\n        out[int(f)] = {"train": pool[~pool["channel_id"].isin(chs)].reset_index(drop=True),\n                       "test": gold[gold["channel_id"].isin(chs)].reset_index(drop=True)}\n    return out\n\n\ndef group_valid_split(train: pd.DataFrame, frac: float = 0.1, seed: int = 42) -> tuple[pd.Series, pd.Series]:\n    """Валідація всередині train для Optuna і ранньої зупинки. Групи за text_hash: однакові тексти не розходяться між частинами.\n    Повертає (train_mask, valid_mask) у порядку рядків train."""\n    groups = train["text_hash"].to_numpy()\n    uniq = np.unique(groups)\n    rng = np.random.default_rng(seed)\n    valid_groups = set(rng.choice(uniq, size=max(1, int(round(frac * len(uniq)))), replace=False).tolist())\n    v = pd.Series([g in valid_groups for g in groups], index=train.index)\n    return ~v, v\n\n\ndef channel_prior(train: pd.DataFrame, alpha: float = 1.0) -> pd.Series:\n    """Згладжена частка наративних слабких міток каналу, рахується ЛИШЕ з переданого train. Для каналів, яких там немає,\n    значення відсутнє (викликач підставляє глобальну частку train). Згладжування до глобальної частки: (n_pos + alpha * p) / (n + alpha)."""\n    p = float(train["weak_narrative"].mean())\n    g = train.groupby("channel_id")["weak_narrative"].agg(["sum", "size"])\n    return ((g["sum"] + alpha * p) / (g["size"] + alpha)).rename("channel_prior")\n\n\ndef leakage_checks(train: pd.DataFrame, test: pd.DataFrame, gold_copy_pairs: pd.DataFrame | None = None,\n                   threshold: float = SEM_THRESHOLD, by_channel: bool = False) -> list[dict]:\n    """Перевірки витоку для однієї пари train/test. gold_copy_pairs: gold_key, neighbor_key, cos (S4, повний пошук копій золотих)."""\n    checks = []\n    inter = set(train["key"]) & set(test["key"])\n    checks.append({"check": "Перетин ключів train і test порожній", "passed": not inter, "detail": f"{len(inter)} спільних ключів"})\n    if "text_hash" in train and "text_hash" in test:\n        ht = set(test["text_hash"].dropna())\n        n_same = int(train["text_hash"].isin(ht).sum())\n        checks.append({"check": "Немає train-повідомлень з тим самим нормалізованим текстом, що й у test", "passed": n_same == 0, "detail": f"{n_same} збігів"})\n    if gold_copy_pairs is not None and len(gold_copy_pairs):\n        gp = gold_copy_pairs[(gold_copy_pairs["cos"] >= threshold) & gold_copy_pairs["gold_key"].isin(set(test["key"]))]\n        n_bad = int(gp["neighbor_key"].isin(set(train["key"])).sum())\n        checks.append({"check": f"Немає train-повідомлень зі схожістю до test не нижче {threshold}".replace(".", ","), "passed": n_bad == 0,\n                       "detail": f"{n_bad} пар (з {len(gp)} пар копій test)"})\n    if by_channel:\n        both = set(train["channel_id"]) & set(test["channel_id"])\n        checks.append({"check": "Канали train і test не перетинаються", "passed": not both, "detail": f"{len(both)} спільних каналів"})\n    return checks\n\n\ndef size_row(protocol: str, fold: int | None, train: pd.DataFrame, test: pd.DataFrame) -> dict:\n    return {"protocol": protocol, "fold": fold, "n_train": int(len(train)), "n_train_narrative": int(train["weak_narrative"].sum()),\n            "n_train_channels": int(train["channel_id"].nunique()), "n_test": int(len(test)), "n_test_narrative": int(test["y"].sum()),\n            "n_test_channels": int(test["channel_id"].nunique())}\n\n\n# ---------------------------------------------------------------- розмітка «явний / неявний» (визначення а)\nEXPL_VALUES = ("explicit", "implicit", "unclear")\nREASON_CODES = ("direct", "framing", "irony", "quote", "context", "fragment", "mixed", "unrelated")\nMIN_IMPLICIT = 30\n\n\ndef implicit_manual_summary(lab: pd.DataFrame, min_implicit: int = MIN_IMPLICIT) -> dict:\n    """Підсумок ручної розмітки. Коментар має вигляд `код` або `код;borderline`. Чутливість: вузьке (без borderline) і широке (borderline явних вважаємо неявними) визначення."""\n    ex = lab["explicitness"].fillna("").astype(str).str.strip()\n    bad = sorted(set(ex) - set(EXPL_VALUES))\n    code = lab["comment"].fillna("").astype(str).str.split(";").str[0].str.strip()\n    border = lab["comment"].fillna("").astype(str).str.contains("borderline")\n    bad_code = sorted(set(code) - set(REASON_CODES))\n    ru = lab["lang"].eq("ru")\n\n    def cnt(mask):\n        return {"all": int(mask.sum()), "ru": int((mask & ru).sum()), "uk": int((mask & lab["lang"].eq("uk")).sum())}\n\n    imp, exp, unc = ex.eq("implicit"), ex.eq("explicit"), ex.eq("unclear")\n    narrow, wide = imp & ~border, imp | (exp & border)\n    out = {"n_rows": int(len(lab)), "n_filled": int(ex.ne("").sum()), "bad_values": bad, "bad_codes": bad_code,\n           "explicit": cnt(exp), "implicit": cnt(imp), "unclear": cnt(unc),\n           "implicit_narrow": cnt(narrow), "implicit_wide": cnt(wide), "n_borderline": int(border.sum()),\n           "by_code": {k: int((code == k).sum()) for k in REASON_CODES if (code == k).any()},\n           "min_implicit": int(min_implicit),\n           "enough_ru": {"base": bool((imp & ru).sum() >= min_implicit), "narrow": bool((narrow & ru).sum() >= min_implicit),\n                         "wide": bool((wide & ru).sum() >= min_implicit)}}\n    return out\n\n\ndef spotcheck_sample(lab: pd.DataFrame, n_per_class: int = 10, seed: int = 42) -> pd.DataFrame:\n    """Сліпа вибірка для перевірки користувачем: n_per_class явних і n_per_class неявних (за розміткою Claude), без самої розмітки."""\n    rng = np.random.default_rng(seed)\n    pick = []\n    for v in ("explicit", "implicit"):\n        ids = lab.loc[lab["explicitness"] == v, "n"].to_numpy()\n        pick += list(rng.choice(ids, size=min(n_per_class, len(ids)), replace=False))\n    sub = lab[lab["n"].isin(pick)].sample(frac=1.0, random_state=seed)\n    return sub[["n", "narrative_label", "lang", "text"]].assign(explicitness_user="").reset_index(drop=True)\n\n\ndef spotcheck_agreement(lab: pd.DataFrame, user: pd.DataFrame) -> dict | None:\n    """Узгодженість розмітки Claude з перевіркою користувача на сліпій вибірці. None, якщо відповідей ще немає."""\n    u = user.copy()\n    u["explicitness_user"] = u["explicitness_user"].fillna("").astype(str).str.strip()\n    u = u[u["explicitness_user"].ne("")]\n    if u.empty:\n        return None\n    m = u.merge(lab[["n", "explicitness"]], on="n")\n    both = m[m["explicitness_user"].isin(("explicit", "implicit")) & m["explicitness"].isin(("explicit", "implicit"))]\n    return {"n_checked": int(len(m)), "n_binary": int(len(both)), "n_agree": int((both["explicitness_user"] == both["explicitness"]).sum()),\n            "n_user_unclear": int((m["explicitness_user"] == "unclear").sum())}\n', 'src/narr/textmodels.py': '"""S6: тренування текстових моделей на пулі зі слабкими мітками і прогнози для золотих повідомлень.\n\nМоделі: `tfidf` (слова 1-2 + символьні 2-5 n-грами, LogReg), `bge_lr` (ембединги bge-m3 з S4, LogReg), `xlmr` (дотренування XLM-R base).\nКонтролі з перемішаними мітками: `tfidf_shuf`, `bge_lr_shuf`, `xlmr_shuf` (лише P1).\n\nПравила, що не залежать від золотих:\n- валідація всередині train: 10% за групами text_hash, seed 42 (`splits.group_valid_split`);\n- C підбирається за macro-F1 на цій валідації (поріг 0,5, слабкі мітки), навчання лише на 90% train, без повторного навчання на всьому train;\n- C підбирається на P1; для P2 і P3 береться те саме C (економія часу, значення пишеться в unit-файли);\n- XLM-R: епоха обирається за валідаційним macro-F1; золоті мітки в цьому модулі не використовуються.\nРезультат одиниці (модель, протокол, seed): `predictions/<unit>.parquet` (ймовірності для золотих тесту) і `units/<unit>.json` (валідація, C, тривалість).\nЯкщо обидва файли вже є, одиницю пропускаємо (можна відновлювати перервані запуски).\n"""\nfrom __future__ import annotations\n\nimport json\nimport time\nimport warnings\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom scipy import sparse\nfrom sklearn.linear_model import LogisticRegression\n\nfrom narr.baselines import _char, _word\nfrom narr.evaluate import binary_metrics\nfrom narr.splits import group_valid_split, make_key\n\nSEED = 42\nVALID_FRAC = 0.1\nCS_TFIDF = (2.0, 8.0, 32.0)\nCS_DENSE = (3.0, 30.0, 300.0)\nTEXT_MODELS = ("tfidf", "bge_lr", "xlmr")\nSHUF_SUFFIX = "_shuf"\n\n\ndef hash_texts(texts: pd.Series) -> pd.Series:\n    """Той самий хеш, що в `semantic.build_embedding_set` (за ним S4 зберіг `text_hash`)."""\n    return pd.util.hash_pandas_object(texts.astype(str).reset_index(drop=True), index=False).astype("uint64").map(lambda v: f"{int(v):016x}").set_axis(texts.index)\n\n\n# ---------------------------------------------------------------- спліти і тексти\n\ndef load_splits(split_dir: Path | str, protocols=("P1", "P2", "P3")) -> list[dict]:\n    """Список сплітів: {"protocol", "fold", "train" (channel_id, message_id, weak_narrative, key), "test" (channel_id, message_id, y, date_utc, key)}.\n    P1 і P2 мають один сплiт (fold 0), P3 один на кожну групу каналів (fold 1..K)."""\n    d = Path(split_dir)\n    out = []\n    for pr in protocols:\n        if pr in ("P1", "P2"):\n            pairs = [(0, d / pr / "train.csv.gz", d / pr / "test.csv")]\n        elif pr == "P3":\n            ks = sorted(int(p.name.split("_")[0][4:]) for p in (d / "P3").glob("fold*_train.csv.gz"))\n            pairs = [(k, d / "P3" / f"fold{k}_train.csv.gz", d / "P3" / f"fold{k}_test.csv") for k in ks]\n        else:\n            raise ValueError(pr)\n        for k, ptr, pte in pairs:\n            tr, te = pd.read_csv(ptr), pd.read_csv(pte)\n            tr["weak_narrative"] = tr["weak_narrative"].astype(bool)\n            tr["key"] = make_key(tr["channel_id"], tr["message_id"])\n            te["key"] = make_key(te["channel_id"], te["message_id"])\n            out.append({"protocol": pr, "fold": k, "train": tr, "test": te})\n    return out\n\n\ndef needed_keys(splits: list[dict]) -> np.ndarray:\n    return np.unique(np.concatenate([np.concatenate([s["train"]["key"].to_numpy(), s["test"]["key"].to_numpy()]) for s in splits]))\n\n\ndef load_texts(raw_dir: Path | str, keys: np.ndarray) -> pd.Series:\n    """Нормалізовані тексти з telegram_messages.parquet для потрібних ключів (як у S4). Повертає Series key -> text."""\n    import pyarrow.parquet as pq\n\n    tg = pq.read_table(Path(raw_dir) / "telegram_messages.parquet", columns=["channel_id", "message_id", "text"]).to_pandas()\n    tg["key"] = make_key(tg["channel_id"], tg["message_id"])\n    tg = tg[tg["key"].isin(set(keys.tolist()))].drop_duplicates("key")\n    s = pd.Series(tg["text"].to_numpy(), index=tg["key"].to_numpy())\n    missing = np.setdiff1d(keys, s.index.to_numpy())\n    if len(missing):\n        raise KeyError(f"для {len(missing)} ключів немає тексту в telegram_messages.parquet, наприклад {missing[:3].tolist()}")\n    return s.loc[keys]\n\n\ndef attach_text(splits: list[dict], texts: pd.Series) -> None:\n    """Додає колонки text і text_hash до train і test кожного спліту (на місці)."""\n    th = hash_texts(texts)\n    for s in splits:\n        for part in ("train", "test"):\n            df = s[part]\n            df["text"] = df["key"].map(texts)\n            df["text_hash"] = df["key"].map(th)\n            if df["text"].isna().any():\n                raise KeyError(f"{s[\'protocol\']} fold {s[\'fold\']} {part}: є повідомлення без тексту")\n\n\ndef check_hash_against_index(texts: pd.Series, index: pd.DataFrame) -> dict:\n    """Звіряє хеш завантаженого тексту з text_hash із індексу ембедингів S4: так перевіряємо, що ембединг і текст належать одному повідомленню."""\n    th = hash_texts(texts)\n    ref = index.drop_duplicates("key").set_index("key")["text_hash"]\n    common = th.index.intersection(ref.index)\n    mism = (th.loc[common] != ref.loc[common]).sum()\n    return {"n_keys": int(len(th)), "n_in_index": int(len(common)), "n_mismatch": int(mism), "ok": bool(len(common) == len(th) and mism == 0)}\n\n\n# ---------------------------------------------------------------- лінійні моделі\n\ndef fit_logreg(X, y: np.ndarray, C: float, seed: int, max_iter: int = 500) -> LogisticRegression:\n    m = LogisticRegression(C=C, class_weight="balanced", max_iter=max_iter, solver="lbfgs", random_state=seed)\n    with warnings.catch_warnings():\n        warnings.simplefilter("ignore")\n        m.fit(X, y)\n    return m\n\n\ndef select_C(Xtr, ytr: np.ndarray, Xva, yva: np.ndarray, Cs, seed: int, max_iter: int = 500, log=print) -> dict:\n    """Перебір C за macro-F1 на валідації (слабкі мітки). Нічия: менше C. Повертає {"model", "C", "valid_macro_f1", "grid"}."""\n    best, grid = None, []\n    for C in Cs:\n        t0 = time.time()\n        m = fit_logreg(Xtr, ytr, C, seed, max_iter)\n        f1 = binary_metrics(yva, m.predict_proba(Xva)[:, 1])["macro_f1"]\n        grid.append({"C": float(C), "valid_macro_f1": float(f1), "n_iter": int(np.max(m.n_iter_)), "seconds": round(time.time() - t0, 1)})\n        log(f"    C={C:g}: valid macro-F1 {f1:.4f} (ітерацій {grid[-1][\'n_iter\']}, {grid[-1][\'seconds\']} с)")\n        if best is None or f1 > best[0] + 1e-12:\n            best = (f1, C, m)\n    return {"model": best[2], "C": float(best[1]), "valid_macro_f1": float(best[0]), "grid": grid}\n\n\ndef _tfidf_features(tr_texts, other: dict[str, list[str]]):\n    w, c = _word(), _char()\n    Xtr = sparse.hstack([w.fit_transform(tr_texts), c.fit_transform(tr_texts)]).tocsr()\n    return Xtr, {k: sparse.hstack([w.transform(v), c.transform(v)]).tocsr() for k, v in other.items()}\n\n\ndef _pred_frame(split: dict, model: str, seed: int, p: np.ndarray) -> pd.DataFrame:\n    te = split["test"]\n    return pd.DataFrame({"protocol": split["protocol"], "fold": split["fold"], "model": model, "seed": seed,\n                         "channel_id": te["channel_id"].to_numpy(), "message_id": te["message_id"].to_numpy(), "p": np.asarray(p, dtype=float)})\n\n\ndef _train_view(split: dict, shuffle_seed: int | None):\n    """Навчальна і валідаційна частини спліту. Для контролю мітки перемішуються в усьому train ще до поділу (валідація теж перемішана)."""\n    tr = split["train"]\n    y = tr["weak_narrative"].astype(int).to_numpy()\n    if shuffle_seed is not None:\n        y = np.random.default_rng(shuffle_seed).permutation(y)\n    m_tr, m_va = group_valid_split(tr, VALID_FRAC, SEED)\n    return tr[m_tr.to_numpy()], tr[m_va.to_numpy()], y[m_tr.to_numpy()], y[m_va.to_numpy()]\n\n\ndef run_lr(kind: str, splits: list[dict], emb_lookup, fixed_C: float | None, shuffle_seed: int | None, log=print) -> tuple[pd.DataFrame, dict]:\n    """Одна одиниця для `tfidf` або `bge_lr` на всіх сплітах одного протоколу."""\n    name = kind + (SHUF_SUFFIX if shuffle_seed is not None else "")\n    preds, per_split = [], []\n    C_used = fixed_C\n    for s in splits:\n        t0 = time.time()\n        a, v, ya, yv = _train_view(s, shuffle_seed)\n        log(f"  {name} {s[\'protocol\']} fold {s[\'fold\']}: train {len(a)}, valid {len(v)}, test {len(s[\'test\'])}")\n        if kind == "tfidf":\n            Xa, other = _tfidf_features(a["text"].tolist(), {"v": v["text"].tolist(), "t": s["test"]["text"].tolist()})\n            Xv, Xt = other["v"], other["t"]\n            grid = CS_TFIDF\n        else:\n            Xa, Xv, Xt = (emb_lookup(d["key"].to_numpy()) for d in (a, v, s["test"]))\n            grid = CS_DENSE\n        res = select_C(Xa, ya, Xv, yv, [C_used] if C_used is not None else grid, SEED, log=log)\n        C_used = res["C"]  # якщо C не було задано, його підбирає перший спліт, решта (фолди P3) беруть те саме\n        p = res["model"].predict_proba(Xt)[:, 1]\n        preds.append(_pred_frame(s, name, 0, p))\n        per_split.append({"fold": s["fold"], "n_train": int(len(a)), "n_valid": int(len(v)), "C": res["C"], "valid_macro_f1": res["valid_macro_f1"],\n                          "grid": res["grid"], "seconds": round(time.time() - t0, 1)})\n    info = {"model": name, "protocol": splits[0]["protocol"], "seed": 0, "C": per_split[0]["C"],\n            "valid_macro_f1": float(np.mean([r["valid_macro_f1"] for r in per_split])), "splits": per_split}\n    return pd.concat(preds, ignore_index=True), info\n\n\ndef run_xlmr(splits: list[dict], seed: int, shuffle_seed: int | None, kwargs: dict, log=print) -> tuple[pd.DataFrame, dict]:\n    from narr.xlmr import fine_tune\n\n    name = "xlmr" + (SHUF_SUFFIX if shuffle_seed is not None else "")\n    preds, per_split = [], []\n    for s in splits:\n        t0 = time.time()\n        a, v, ya, yv = _train_view(s, shuffle_seed)\n        log(f"  {name} {s[\'protocol\']} fold {s[\'fold\']} seed {seed}: train {len(a)}, valid {len(v)}, test {len(s[\'test\'])}")\n        r = fine_tune(a["text"].tolist(), ya, v["text"].tolist(), yv, {"test": s["test"]["text"].tolist()}, seed=seed, log=log, **kwargs)\n        preds.append(_pred_frame(s, name, seed, r["probs"]["test"]))\n        per_split.append({"fold": s["fold"], "n_train": r["n_train"], "n_valid": r["n_valid"], "best_epoch": r["best_epoch"],\n                          "valid_macro_f1": r["valid_macro_f1"], "history": r["history"], "tokens_median": r["tokens_median"],\n                          "share_truncated": r["share_truncated"], "seconds": round(time.time() - t0, 1)})\n    info = {"model": name, "protocol": splits[0]["protocol"], "seed": seed, "valid_macro_f1": float(np.mean([r["valid_macro_f1"] for r in per_split])),\n            "splits": per_split}\n    return pd.concat(preds, ignore_index=True), info\n\n\n# ---------------------------------------------------------------- оркестрація з відновленням\n\ndef unit_name(model: str, protocol: str, seed: int) -> str:\n    return f"{model}__{protocol}__s{seed}"\n\n\ndef run_all(splits: list[dict], out: Path | str, *, models=TEXT_MODELS, protocols=("P1", "P2", "P3"), xlmr_protocols=("P1", "P2"),\n            seeds=(0, 1, 2), controls=True, emb_lookup=None, xlmr_kwargs: dict | None = None, log=print) -> list[dict]:\n    """Проходить усі одиниці у порядку: P1 перед P2 і P3 (щоб C для P2, P3 взяти з P1). Повертає список info усіх одиниць (нових і з диска)."""\n    out = Path(out)\n    (out / "units").mkdir(parents=True, exist_ok=True)\n    (out / "predictions").mkdir(parents=True, exist_ok=True)\n    by_pr = {pr: [s for s in splits if s["protocol"] == pr] for pr in protocols}\n    infos, chosen_C = [], {}\n\n    def unit(model: str, protocol: str, seed: int, fn):\n        un = unit_name(model, protocol, seed)\n        pj, pp = out / "units" / f"{un}.json", out / "predictions" / f"{un}.parquet"\n        if pj.exists() and pp.exists():\n            info = json.loads(pj.read_text(encoding="utf-8"))\n            log(f"[пропуск] {un}: уже є")\n        else:\n            log(f"[{time.strftime(\'%H:%M:%S\')}] {un}")\n            t0 = time.time()\n            pred, info = fn()\n            info["seconds"] = round(time.time() - t0, 1)\n            pred.to_parquet(pp, index=False)\n            pj.write_text(json.dumps(info, ensure_ascii=False, indent=1), encoding="utf-8")\n            log(f"  готово за {info[\'seconds\']} с, valid macro-F1 {info[\'valid_macro_f1\']:.4f}")\n        infos.append(info)\n        return info\n\n    for kind in [m for m in models if m in ("tfidf", "bge_lr")]:\n        for pr in protocols:  # порядок P1, P2, P3 береться з аргументу, P1 має йти першим\n            info = unit(kind, pr, 0, lambda: run_lr(kind, by_pr[pr], emb_lookup, chosen_C.get(kind), None, log))\n            chosen_C.setdefault(kind, info["C"])\n    if "xlmr" in models:\n        for pr in [p for p in protocols if p in xlmr_protocols]:\n            for sd in seeds:\n                unit("xlmr", pr, sd, lambda: run_xlmr(by_pr[pr], sd, None, xlmr_kwargs or {}, log))\n    if controls and "P1" in by_pr:\n        for kind in [m for m in models if m in ("tfidf", "bge_lr")]:\n            unit(kind + SHUF_SUFFIX, "P1", 0, lambda: run_lr(kind, by_pr["P1"], emb_lookup, chosen_C.get(kind), 1234, log))\n        if "xlmr" in models and "P1" in xlmr_protocols:\n            unit("xlmr" + SHUF_SUFFIX, "P1", 0, lambda: run_xlmr(by_pr["P1"], 0, 1234, {**(xlmr_kwargs or {}), "epochs": 1}, log))\n    return infos\n', 'src/narr/xlmr.py': '"""S6: дотренування XLM-R на пулі зі слабкими мітками (бінарна задача «наратив / ні»).\n\ntorch і transformers імпортуються лише всередині функцій, тож модуль підключається і в середовищі без них (хмарна сесія).\nЛогіка без torch (пакування за довжиною, вагові коефіцієнти класів, розклад швидкості навчання) винесена в окремі функції і тестується.\n\nПротокол: тренуємо на частині пулу `train`, після кожної епохи рахуємо macro-F1 на валідації зі слабкими мітками (поріг 0,5).\nПрогнози для тестових текстів (золоті) беруться з епохи з найкращою валідацією. Золоті мітки в цьому модулі не з\'являються взагалі:\nфункція отримує лише тексти золотих, без міток.\n"""\nfrom __future__ import annotations\n\nimport math\nimport random\nimport time\n\nimport numpy as np\n\nfrom narr.evaluate import binary_metrics\n\n\ndef length_bucket_batches(lengths: np.ndarray, batch_size: int, rng: np.random.Generator, chunk_mult: int = 50) -> list[np.ndarray]:\n    """Перемішує приклади, у блоках по batch_size*chunk_mult сортує за довжиною (менше доповнень), ріже на партії і перемішує партії."""\n    lengths = np.asarray(lengths)\n    idx = rng.permutation(len(lengths))\n    chunk = batch_size * chunk_mult\n    batches: list[np.ndarray] = []\n    for s in range(0, len(idx), chunk):\n        c = idx[s:s + chunk]\n        c = c[np.argsort(lengths[c], kind="stable")]\n        batches += [c[i:i + batch_size] for i in range(0, len(c), batch_size)]\n    return [batches[i] for i in rng.permutation(len(batches))]\n\n\ndef pad_batch(seqs: list[list[int]], pad_id: int) -> tuple[np.ndarray, np.ndarray]:\n    """Доповнює послідовності до найдовшої в партії. Повертає (input_ids, attention_mask)."""\n    width = max(len(s) for s in seqs)\n    ids = np.full((len(seqs), width), pad_id, dtype=np.int64)\n    mask = np.zeros((len(seqs), width), dtype=np.int64)\n    for r, s in enumerate(seqs):\n        ids[r, :len(s)] = s\n        mask[r, :len(s)] = 1\n    return ids, mask\n\n\ndef class_weights(y: np.ndarray) -> np.ndarray:\n    """Зважування класів n / (2 * n_c), як class_weight=\'balanced\' у лінійних моделях."""\n    y = np.asarray(y).astype(int)\n    cnt = np.bincount(y, minlength=2).astype(float)\n    if (cnt == 0).any():\n        raise ValueError("в навчальній частині немає одного з класів")\n    return len(y) / (2.0 * cnt)\n\n\ndef lr_factor(step: int, total: int, warmup: int) -> float:\n    """Лінійний розігрів до 1, далі лінійне спадання до 0."""\n    if step < warmup:\n        return step / max(1, warmup)\n    return max(0.0, (total - step) / max(1, total - warmup))\n\n\ndef encode_texts(tokenizer, texts: list[str], max_len: int, chunk: int = 5000) -> list[list[int]]:\n    out: list[list[int]] = []\n    for s in range(0, len(texts), chunk):\n        out += tokenizer(list(texts[s:s + chunk]), truncation=True, max_length=max_len)["input_ids"]\n    return out\n\n\ndef _seed_everything(seed: int) -> None:\n    import torch\n\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n\n\ndef predict_proba(model, ids: list[list[int]], pad_id: int, device, batch_size: int = 128, amp: bool = False) -> np.ndarray:\n    """P(клас 1) для кожної послідовності, у початковому порядку (всередині сортуємо за довжиною)."""\n    import torch\n\n    order = np.argsort([len(s) for s in ids], kind="stable")\n    out = np.zeros(len(ids), dtype=np.float64)\n    model.eval()\n    with torch.no_grad():\n        for s in range(0, len(ids), batch_size):\n            b = order[s:s + batch_size]\n            x, m = pad_batch([ids[i] for i in b], pad_id)\n            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):\n                logits = model(input_ids=torch.as_tensor(x, device=device), attention_mask=torch.as_tensor(m, device=device)).logits\n            out[b] = torch.softmax(logits.float(), dim=-1)[:, 1].cpu().numpy()\n    return out\n\n\ndef fine_tune(train_texts: list[str], y_train: np.ndarray, valid_texts: list[str], y_valid: np.ndarray,\n              eval_texts: dict[str, list[str]], *, seed: int, epochs: int = 3, lr: float = 2e-5, batch_size: int = 32,\n              max_len: int = 128, warmup_frac: float = 0.06, weight_decay: float = 0.01, model_name: str = "xlm-roberta-base",\n              device: str | None = None, fp16: bool = True, tokenizer=None, model_factory=None, log=print) -> dict:\n    """Дотренування з вибором епохи за валідаційним macro-F1 (слабкі мітки).\n\n    Повертає {"valid_macro_f1", "best_epoch", "history", "probs": {назва: ймовірності з найкращої епохи}, ...}.\n    `tokenizer` і `model_factory` можна підставити (тести); за замовчуванням завантажується `model_name` з HuggingFace."""\n    import torch\n    from torch.optim.lr_scheduler import LambdaLR\n\n    y_train, y_valid = np.asarray(y_train).astype(int), np.asarray(y_valid).astype(int)\n    _seed_everything(seed)\n    dev = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))\n    amp = bool(fp16 and dev.type == "cuda")\n    if tokenizer is None:\n        from transformers import AutoTokenizer\n        tokenizer = AutoTokenizer.from_pretrained(model_name)\n    if model_factory is None:\n        from transformers import AutoModelForSequenceClassification\n\n        def model_factory():\n            return AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)\n    pad_id = int(tokenizer.pad_token_id)\n\n    t0 = time.time()\n    ids_tr = encode_texts(tokenizer, train_texts, max_len)\n    ids_va = encode_texts(tokenizer, valid_texts, max_len)\n    ids_ev = {k: encode_texts(tokenizer, v, max_len) for k, v in eval_texts.items()}\n    len_tr = np.array([len(s) for s in ids_tr])\n    info = {"n_train": int(len(ids_tr)), "n_valid": int(len(ids_va)), "max_len": int(max_len), "tokens_median": float(np.median(len_tr)),\n            "share_truncated": float((len_tr >= max_len).mean()), "tokenize_seconds": round(time.time() - t0, 1)}\n    log(f"  токенізація {info[\'tokenize_seconds\']} с, медіана {info[\'tokens_median\']:.0f} токенів, обрізано {info[\'share_truncated\']:.1%}")\n\n    model = model_factory().to(dev)\n    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)\n    steps_per_epoch = math.ceil(len(ids_tr) / batch_size)\n    total, warm = steps_per_epoch * epochs, int(warmup_frac * steps_per_epoch * epochs)\n    sched = LambdaLR(opt, lambda s: lr_factor(s, total, warm))\n    scaler = torch.amp.GradScaler("cuda", enabled=amp)\n    loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor(class_weights(y_train), dtype=torch.float32, device=dev))\n    rng = np.random.default_rng(seed)\n\n    best = {"valid_macro_f1": -1.0, "best_epoch": -1, "probs": {}}\n    history = []\n    for epoch in range(1, epochs + 1):\n        model.train()\n        t_ep, losses = time.time(), []\n        for b in length_bucket_batches(len_tr, batch_size, rng):\n            x, m = pad_batch([ids_tr[i] for i in b], pad_id)\n            xb, mb = torch.as_tensor(x, device=dev), torch.as_tensor(m, device=dev)\n            yb = torch.as_tensor(y_train[b], device=dev)\n            opt.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=amp):\n                logits = model(input_ids=xb, attention_mask=mb).logits\n            loss = loss_fn(logits.float(), yb)\n            if not torch.isfinite(loss):\n                raise FloatingPointError(f"нескінченна втрата на епосі {epoch}")\n            scaler.scale(loss).backward()\n            scaler.unscale_(opt)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            scaler.step(opt)\n            scaler.update()\n            sched.step()\n            losses.append(float(loss.detach()))\n        pv = predict_proba(model, ids_va, pad_id, dev, amp=amp)\n        f1 = binary_metrics(y_valid, pv)["macro_f1"]\n        history.append({"epoch": epoch, "train_loss": float(np.mean(losses)), "valid_macro_f1": float(f1), "seconds": round(time.time() - t_ep, 1)})\n        log(f"  епоха {epoch}: втрата {history[-1][\'train_loss\']:.4f}, valid macro-F1 {f1:.4f}, {history[-1][\'seconds\']} с")\n        if f1 > best["valid_macro_f1"]:\n            best = {"valid_macro_f1": float(f1), "best_epoch": epoch,\n                    "probs": {k: predict_proba(model, v, pad_id, dev, amp=amp) for k, v in ids_ev.items()}}\n    del model, opt\n    if dev.type == "cuda":\n        torch.cuda.empty_cache()\n    return {**best, "history": history, **info}\n', 'scripts/s06_train.py': '"""S6 (Colab): навчання текстових моделей на пулі зі слабкими мітками, прогнози для золотих повідомлень за P1-P3.\n\nМоделі: TF-IDF + LogReg, ембединги bge-m3 (кеш S4) + LogReg, XLM-R base (дотренування), контролі з перемішаними мітками.\nМетрики тут не рахуються: прогнози і валідація зберігаються, оцінка на золотих у `scripts/s06_eval.py` (хмарна сесія).\n\nЗапуск (Colab): див. colab/06_text_models.ipynb. Приклад:\n  python3 -I scripts/s06_train.py --raw-dir <data/raw> --splits-dir <splits> --emb-dir <data/interim/embeddings> --out <results/06_text_models>\nСамоперевірка XLM-R на крихітній випадковій моделі без завантаження ваг (секунди):  python3 -I scripts/s06_train.py --selftest\n"""\nimport argparse\nimport json\nimport sys\nimport time\nimport zlib\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))\n\nimport numpy as np  # noqa: E402\nimport pandas as pd  # noqa: E402\n\nfrom narr import textmodels as tm  # noqa: E402\n\n\ndef selftest() -> None:\n    """Проганяє справжній цикл `fine_tune` (torch + transformers) на синтетичних даних з крихітною випадковою моделлю."""\n    from transformers import XLMRobertaConfig, XLMRobertaForSequenceClassification\n\n    from narr.xlmr import fine_tune\n\n    class Tok:\n        pad_token_id = 1\n\n        def __call__(self, texts, truncation=True, max_length=64):\n            return {"input_ids": [[0] + [zlib.crc32(w.encode()) % 480 + 5 for w in t.split()][:max_length - 2] + [2] for t in texts]}\n\n    rng = np.random.default_rng(0)\n    words = [f"w{i}" for i in range(40)]\n\n    def make(n):\n        y = rng.integers(0, 2, n)\n        texts = []\n        for v in y:\n            w = rng.choice(words, 6).tolist()\n            if v:\n                w.insert(int(rng.integers(0, 7)), "ключ")  # сигнал на випадковій позиції\n            texts.append(" ".join(w))\n        return texts, y\n\n    xtr, ytr = make(480)\n    xva, yva = make(100)\n    xte, _ = make(20)\n\n    def factory():\n        cfg = XLMRobertaConfig(vocab_size=500, hidden_size=32, num_hidden_layers=1, num_attention_heads=2, intermediate_size=64,\n                               max_position_embeddings=140, num_labels=2, pad_token_id=1)\n        return XLMRobertaForSequenceClassification(cfg)\n\n    r = fine_tune(xtr, ytr, xva, yva, {"gold": xte}, seed=0, epochs=6, lr=1e-3, batch_size=16, max_len=32, tokenizer=Tok(), model_factory=factory)\n    assert len(r["probs"]["gold"]) == 20 and np.isfinite(r["probs"]["gold"]).all()\n    assert r["valid_macro_f1"] > 0.8, f"самоперевірка: крихітна модель не навчилася (valid macro-F1 {r[\'valid_macro_f1\']:.3f})"\n    print(f"САМОПЕРЕВІРКА ПРОЙДЕНА: valid macro-F1 {r[\'valid_macro_f1\']:.3f}, епоха {r[\'best_epoch\']}")\n\n\ndef make_lookup(emb_dir: Path):\n    keys = np.load(emb_dir / "bge_m3_keys.npy")\n    emb = np.load(emb_dir / "bge_m3_fp16.npy", mmap_mode="r")\n    row = pd.Series(np.arange(len(keys)), index=keys)\n\n    def lookup(k: np.ndarray) -> np.ndarray:\n        v = np.asarray(emb[row.loc[k].to_numpy()], dtype=np.float32)\n        return v / np.maximum(np.linalg.norm(v, axis=1, keepdims=True), 1e-12)\n\n    return lookup\n\n\ndef main(argv=None) -> int:\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--selftest", action="store_true")\n    ap.add_argument("--raw-dir")\n    ap.add_argument("--splits-dir")\n    ap.add_argument("--emb-dir")\n    ap.add_argument("--out")\n    ap.add_argument("--models", default="tfidf,bge_lr,xlmr")\n    ap.add_argument("--protocols", default="P1,P2,P3")\n    ap.add_argument("--xlmr-protocols", default="P1,P2")\n    ap.add_argument("--seeds", type=int, default=3)\n    ap.add_argument("--epochs", type=int, default=3)\n    ap.add_argument("--max-len", type=int, default=128)\n    ap.add_argument("--batch-size", type=int, default=32)\n    ap.add_argument("--lr", type=float, default=2e-5)\n    ap.add_argument("--model-name", default="xlm-roberta-base")\n    ap.add_argument("--no-controls", action="store_true")\n    ap.add_argument("--limit", type=int, default=0, help="смоук-тест: залишити не більше N рядків train у кожному спліті")\n    args = ap.parse_args(argv)\n    if args.selftest:\n        selftest()\n        return 0\n    t0 = time.time()\n    out = Path(args.out)\n    out.mkdir(parents=True, exist_ok=True)\n    models = tuple(args.models.split(","))\n    protocols = tuple(args.protocols.split(","))\n\n    splits = tm.load_splits(args.splits_dir, protocols)\n    keys = tm.needed_keys(splits)\n    print(f"Спліти: {len(splits)}, потрібних ключів {len(keys)}", flush=True)\n    texts = tm.load_texts(args.raw_dir, keys)\n    tm.attach_text(splits, texts)\n    index = pd.read_parquet(Path(args.emb_dir) / "bge_m3_index.parquet", columns=["key", "text_hash"])\n    integrity = tm.check_hash_against_index(texts, index)\n    print("Звірка хешів текстів з індексом ембедингів S4:", integrity, flush=True)\n    if not integrity["ok"]:\n        raise SystemExit("Тексти не збігаються з тими, що кодувалися в S4: зупиняюсь.")\n    for s in splits:  # золоті не в train (страхувальна перевірка, основна в S5)\n        assert not set(s["train"]["key"]) & set(s["test"]["key"]), f"перетин train і test у {s[\'protocol\']} fold {s[\'fold\']}"\n        if args.limit:\n            s["train"] = s["train"].sample(n=min(args.limit, len(s["train"])), random_state=42).reset_index(drop=True)\n\n    xk = dict(epochs=args.epochs, max_len=args.max_len, batch_size=args.batch_size, lr=args.lr, model_name=args.model_name)\n    lookup = make_lookup(Path(args.emb_dir)) if "bge_lr" in models else None\n    infos = tm.run_all(splits, out, models=models, protocols=protocols, xlmr_protocols=tuple(args.xlmr_protocols.split(",")),\n                       seeds=tuple(range(args.seeds)), controls=not args.no_controls, emb_lookup=lookup, xlmr_kwargs=xk)\n\n    try:\n        import torch\n        gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None\n        tv = torch.__version__\n    except ImportError:\n        gpu, tv = None, None\n    meta = {"args": vars(args), "integrity": integrity, "n_units": len(infos), "seconds_total": round(time.time() - t0, 1), "gpu": gpu, "torch": tv,\n            "split_sizes": [{"protocol": s["protocol"], "fold": s["fold"], "n_train": int(len(s["train"])), "n_test": int(len(s["test"]))} for s in splits]}\n    (out / "colab_run.json").write_text(json.dumps(meta, ensure_ascii=False, indent=1), encoding="utf-8")\n    print(f"Готово: {len(infos)} одиниць за {meta[\'seconds_total\']} с", flush=True)\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n'}
SPLITS_ZIP_B64 = "..."

for rel, text in FILES.items():
    p = REPO / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text, encoding="utf-8")
    print(f"{hashlib.sha256(text.encode()).hexdigest()[:12]}  {rel}")
with zipfile.ZipFile(io.BytesIO(base64.b64decode(SPLITS_ZIP_B64))) as z:
    z.extractall(REPO / "splits")
print("Спліти:", sorted(str(p.relative_to(REPO / "splits")) for p in (REPO / "splits").rglob("*") if p.is_file()))


In [ ]:
# 3. Перевірка, що потрібні файли на місці
for p in [RAW / "telegram_messages.parquet", EMB / "bge_m3_keys.npy", EMB / "bge_m3_fp16.npy", EMB / "bge_m3_index.parquet"]:
    assert p.exists(), f"немає {p}"
    print("ок", p, f"{p.stat().st_size / 1e6:.0f} МБ")
try:
    import psutil
    print(f"Пам'ять: {psutil.virtual_memory().total / 1e9:.1f} ГБ усього, {psutil.virtual_memory().available / 1e9:.1f} ГБ вільно")
except ImportError:
    pass


In [ ]:
# 4. Самоперевірка циклу XLM-R (крихітна випадкова модель, без завантаження ваг, секунди)
r = subprocess.run([sys.executable, "-I", str(REPO / "scripts" / "s06_train.py"), "--selftest"], capture_output=True, text=True, cwd=str(REPO))
print(r.stdout[-2000:], r.stderr[-3000:])
assert r.returncode == 0, "самоперевірка не пройшла: див. вивід вище, основний запуск не починаємо"


In [ ]:
# 5. Основний запуск (scripts/s06_train.py). Завершені одиниці пропускаються, тож запуск можна повторювати.
XLMR_PROTOCOLS = "P1,P2"   # P3 для XLM-R не запускаємо (5 фолдів x 3 seed надто довго); за браком часу можна лишити "P1"
SEEDS = 3
cmd = [sys.executable, "-I", str(REPO / "scripts" / "s06_train.py"), "--raw-dir", str(RAW), "--splits-dir", str(REPO / "splits"),
       "--emb-dir", str(EMB), "--out", str(OUT), "--models", "tfidf,bge_lr,xlmr", "--protocols", "P1,P2,P3",
       "--xlmr-protocols", XLMR_PROTOCOLS, "--seeds", str(SEEDS), "--epochs", "3", "--max-len", "128", "--batch-size", "32", "--lr", "2e-5"]
t0 = time.time()
code = None
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=str(REPO))
try:
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
except BaseException:  # перерив (Interrupt) або обрив: зупиняємо дочірній процес, щоб він не лишився працювати у фоні
    proc.terminate()
    proc.wait()
    print("\nЗапуск перервано, дочірній процес зупинено. Завершені одиниці збережено, запустіть цю клітинку знову.")
    raise
print(f"\nКод завершення {code}, {time.time() - t0:.0f} с")


In [ ]:
# 6. Підсумок і архів (безпечно запускати і до завершення клітинки 5: тоді архів буде позначено як неповний)
import shutil

units = [json.loads(p.read_text(encoding="utf-8")) for p in sorted((OUT / "units").glob("*.json"))]
n_expected = 3 + 3 + len(XLMR_PROTOCOLS.split(",")) * SEEDS + 3  # tfidf, bge_lr (P1-P3), xlmr (протоколи x seed), три контролі з перемішаними мітками
code = globals().get("code")
complete = code == 0 and len(units) == n_expected
print(f"Одиниць: {len(units)} з {n_expected}")
for u in units:
    print(f"  {u['model']:12s} {u['protocol']} seed {u['seed']}  valid macro-F1 {u['valid_macro_f1']:.4f}  {u.get('seconds', 0):.0f} с")
name = "06_text_models" if complete else "06_text_models_PARTIAL"
shutil.make_archive(str(OUT.parent / name), "zip", OUT)
print("Архів:", OUT.parent / f"{name}.zip")
if not complete:
    print("\nУВАГА: запуск не завершено (код завершення:", code, "). Цей архів неповний, мені його поки не надсилайте: запустіть клітинку 5 знову.")
